In [1]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
import warnings
warnings.filterwarnings('ignore')

from cdrmip_extremes import utils, stats
from cdrmip_extremes.load_processed import load_processed, load_scalar_outputs
from cdrmip_extremes.configs import CDR_MODELS as models, PROC_DIR

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)

In [3]:
expt = '1pctCO2-cdr'

In [4]:
gsat = load_processed(PROC_DIR, 'CDRMIP', 'gsat', model_order=models)
gsat_rolling = load_processed(PROC_DIR, 'CDRMIP', 'gsat_rolling', model_order=models)
crossing_years = load_processed(PROC_DIR, 'CDRMIP', 'crossing_year', model_order=models)
gwl_matching_years = load_processed(PROC_DIR, 'CDRMIP', 'gwl_matching_years', model_order=models)
sat_difference = load_processed(PROC_DIR, 'CDRMIP', 'gwl_reversibility', model_order=models)
final_difference = load_processed(PROC_DIR, 'CDRMIP', 'gwl_differences', model_order=models)

  sort_models: no order defined for ['multi_model_median'], appending at end
  sort_models: no order defined for ['multi_model_median'], appending at end


In [5]:
peak_gsat = utils.flatten_single(
    load_scalar_outputs(PROC_DIR,'CDRMIP','peak_warming')
)
peak_gsat

{'ACCESS-ESM1-5': {'peak': 4.48, 'peak_year': 152},
 'CESM2': {'peak': 5.35, 'peak_year': 150},
 'CNRM-ESM2-1': {'peak': 5.46, 'peak_year': 147},
 'CanESM5': {'peak': 6.78, 'peak_year': 150},
 'GFDL-ESM4': {'peak': 3.77, 'peak_year': 145},
 'MIROC-ES2L': {'peak': 3.65, 'peak_year': 145},
 'NorESM2-LM': {'peak': 3.45, 'peak_year': 148},
 'UKESM1-0-LL': {'peak': 6.71, 'peak_year': 149}}

In [6]:
def calc_gsat_metrics(ds,ds_rolling,peak_gsat,crossing_years):
    peak = ds_rolling.max(dim='year')
    peak_year = peak_gsat['peak_year']
    peak = peak_gsat['peak']
    cross_year_up = int(crossing_years.crossing_year.sel(branch='ramp_up').values)
    cross_year_down = int(crossing_years.crossing_year.sel(branch='ramp_down').values)
    peak_co2_warming = ds_rolling.sel(year=140).values
    return_to_PI_anom = ds_rolling.sel(year=280).values
    warming_rate_co2 = ((peak_co2_warming/140)*10)
    cooling_rate_co2 = (((return_to_PI_anom - peak_co2_warming)/140)*10)
    
    metric_dict = {
        'Peak warming (°C)': np.round(peak,2),
        'Year of peak warming': peak_year,
        'Year of crossing of 1.5°C ramp-up': cross_year_up,
        'Year of crossing of 1.5°C ramp-down': cross_year_down,
        'Warming rate to peak warming (°C/decade)': np.round(((peak/peak_year)*10),2),
        'Warming at peak CO₂ (°C)': np.round(peak_co2_warming,2),
        'Cooling rate from peak warming (°C/decade)': np.round((10*(return_to_PI_anom-peak)/(280-peak_year)),2),
        'Warming rate to peak CO₂ (°C/decade)': np.round(warming_rate_co2,2),
        'Warming rate to peak CO₂ / Cooling rate from peak CO₂ (°C/decade)': np.round(warming_rate_co2/cooling_rate_co2,2),
        'Anomaly upon return to PI concentrations (°C)': np.round(return_to_PI_anom,2),
        'Final 21-year anomaly (°C)': np.round(ds_rolling.sel(year=330).values,2),
    }
    
    return metric_dict

In [7]:
metric_dicts = {}
for model in models:
    metric_dicts[model] = calc_gsat_metrics(
        gsat[model][expt].gsat,
        gsat_rolling[model][expt].gsat_rolling,
        peak_gsat[model],
        crossing_years[model][expt].sel(gwl=1.5)
    )
metric_dicts

{'ACCESS-ESM1-5': {'Peak warming (°C)': 4.48,
  'Year of peak warming': 152,
  'Year of crossing of 1.5°C ramp-up': 59,
  'Year of crossing of 1.5°C ramp-down': 262,
  'Warming rate to peak warming (°C/decade)': 0.29,
  'Warming at peak CO₂ (°C)': 4.39,
  'Cooling rate from peak warming (°C/decade)': -0.27,
  'Warming rate to peak CO₂ (°C/decade)': 0.31,
  'Warming rate to peak CO₂ / Cooling rate from peak CO₂ (°C/decade)': -1.29,
  'Anomaly upon return to PI concentrations (°C)': 0.99,
  'Final 21-year anomaly (°C)': 0.73},
 'CanESM5': {'Peak warming (°C)': 6.78,
  'Year of peak warming': 150,
  'Year of crossing of 1.5°C ramp-up': 45,
  'Year of crossing of 1.5°C ramp-down': 290,
  'Warming rate to peak warming (°C/decade)': 0.45,
  'Warming at peak CO₂ (°C)': 6.58,
  'Cooling rate from peak warming (°C/decade)': -0.39,
  'Warming rate to peak CO₂ (°C/decade)': 0.47,
  'Warming rate to peak CO₂ / Cooling rate from peak CO₂ (°C/decade)': -1.35,
  'Anomaly upon return to PI concentrati

In [8]:
global_df = pd.DataFrame.from_dict(metric_dicts,orient='columns')

In [9]:
ECS = {
    'ACCESS-ESM1-5':3.87,
    'CanESM5':5.63,
    'CESM2':5.18,
    'CNRM-ESM2-1':4.76,
    'GFDL-ESM4':2.62,
    'MIROC-ES2L':2.68,
    'NorESM2-LM':2.54,
    'UKESM1-0-LL':5.34
}

global_df.loc['ECS from Song et al. (2023) (°C)'] = ECS

In [10]:
global_df

,ACCESS-ESM1-5,CanESM5,CESM2,CNRM-ESM2-1,GFDL-ESM4,MIROC-ES2L,NorESM2-LM,UKESM1-0-LL
Peak warming (°C),4.48,6.78,5.35,5.46,3.77,3.65,3.45,6.71
Year of peak warming,152.00,150.00,150.00,147.00,145.00,145.00,148.00,149.00
Year of crossing of 1.5°C ramp-up,59.00,45.00,51.00,59.00,68.00,67.00,71.00,38.00
Year of crossing of 1.5°C ramp-down,262.00,290.00,266.00,271.00,243.00,236.00,214.00,332.00
Warming rate to peak warming (°C/decade),0.29,0.45,0.36,0.37,0.26,0.25,0.23,0.45
Warming at peak CO₂ (°C),4.39,6.58,5.21,5.30,3.69,3.59,3.43,6.56
Cooling rate from peak warming (°C/decade),-0.27,-0.39,-0.31,-0.32,-0.23,-0.24,-0.24,-0.35
Warming rate to peak CO₂ (°C/decade),0.31,0.47,0.37,0.38,0.26,0.26,0.25,0.47
Warming rate to peak CO₂ / Cooling rate from peak CO₂ (°C/decade),-1.29,-1.35,-1.32,-1.28,-1.23,-1.11,-1.11,-1.49
Anomaly upon return to PI concentrations (°C),0.99,1.70,1.26,1.17,0.69,0.36,0.33,2.17
